In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sql("CREATE CATALOG IF NOT EXISTS company_ro")
spark.sql("CREATE SCHEMA IF NOT EXISTS company_ro.gold")

In [0]:
%sql
CREATE OR REPLACE TABLE company_ro.gold.top_companies_by_year
AS
WITH ranked_companies AS (
  SELECT
    an,
    cui,
    denumire,
    judet,
    localitate,
    cod_caen_mfinante,
    denumire_caen,
    grupa_caen,
    cifra_afaceri,
    profit_net,
    pierdere_neta,
    nr_mediu_salariati,
    datorii,
    capitaluri_proprii,
    ROW_NUMBER() OVER (PARTITION BY an ORDER BY cifra_afaceri DESC NULLS LAST) AS rank_by_turnover,
    ROW_NUMBER() OVER (PARTITION BY an ORDER BY profit_net DESC NULLS LAST) AS rank_by_profit,
    ROW_NUMBER() OVER (PARTITION BY an ORDER BY nr_mediu_salariati DESC NULLS LAST) AS rank_by_employees
  FROM company_ro.gold.company_financial_summary
)
SELECT *
FROM ranked_companies
WHERE rank_by_turnover <= 100
   OR rank_by_profit <= 100
   OR rank_by_employees <= 100

In [0]:
%sql
CREATE OR REPLACE TABLE company_ro.gold.location_caen_year_stats
AS
SELECT
  an,
  judet,
  localitate,
  grupa_caen,
  cod_caen_mfinante,
  denumire_caen,
  caen_display_name,
  numar_companii,
  total_cifra_afaceri,
  total_venituri,
  total_cheltuieli,
  total_profit_net,
  total_pierdere_neta,
  total_salariati,
  total_datorii,
  total_capitaluri_proprii,
  avg_cifra_afaceri,
  avg_profit_net,
  avg_salariati
FROM (
  SELECT
    an,
    judet,
    localitate,
    grupa_caen,
    cod_caen_mfinante,
    denumire_caen,
    caen_display_name,
    COUNT(DISTINCT cui) AS numar_companii,
    SUM(cifra_afaceri) AS total_cifra_afaceri,
    SUM(venituri_totale) AS total_venituri,
    SUM(cheltuieli_totale) AS total_cheltuieli,
    SUM(profit_net) AS total_profit_net,
    SUM(pierdere_neta) AS total_pierdere_neta,
    SUM(nr_mediu_salariati) AS total_salariati,
    SUM(datorii) AS total_datorii,
    SUM(capitaluri_proprii) AS total_capitaluri_proprii,
    AVG(cifra_afaceri) AS avg_cifra_afaceri,
    AVG(profit_net) AS avg_profit_net,
    AVG(nr_mediu_salariati) AS avg_salariati
  FROM company_ro.gold.company_financial_summary
  GROUP BY
    an,
    judet,
    localitate,
    grupa_caen,
    cod_caen_mfinante,
    denumire_caen,
    caen_display_name
)

In [0]:
%sql
-- To manually refresh the materialized view:
-- REFRESH MATERIALIZED VIEW company_ro.gold.top_companies_by_year

In [0]:
display(spark.sql("""
SELECT
  an,
  COUNT(*) AS rows,
  MIN(rank_by_turnover) AS best_turnover_rank,
  MAX(rank_by_turnover) AS worst_turnover_rank
FROM company_ro.gold.top_companies_by_year
GROUP BY an
ORDER BY an
"""))